# Lab 05 — Lakeflow Declarative Pipelines
## 01 — Citi Bike Source Preparation

This notebook prepares and profiles the two related Citi Bike GBFS sources used by Lab 05.

### Source A — Streaming source
`station_status`

The live status feed changes over time and contains operational values such as:
- `station_id`
- `num_bikes_available`
- `num_docks_available`
- station operating flags
- `last_reported`

Each API poll is saved as a **new immutable JSON snapshot** under:

```text
/Volumes/<catalog>/<schema>/<volume_name>/landing/station_status/
```

Later, Lakeflow will ingest this growing folder incrementally with Auto Loader.

### Source B — Batch/reference JSON
`station_information`

This feed contains station metadata such as:
- `station_id`
- station name
- latitude / longitude
- capacity

It is saved as a single reference JSON file under:

```text
/Volumes/<catalog>/<schema>/<volume_name>/reference/
```

### Natural relationship

The two sources are related by:

```text
station_status.station_id
        =
station_information.station_id
```

This gives Lab 05 a real enrichment scenario for the Silver layer.

### Responsibilities of this notebook
- Discover the current Citi Bike GBFS endpoints
- Download the current `station_information` reference file
- Create several initial `station_status` snapshot files
- Profile both sources
- Validate that `station_id` can be used as the join key
- Confirm source preparation completed successfully

### This notebook does NOT
- Create Bronze/Silver/Gold tables
- Create Lakeflow pipeline datasets
- Manage checkpoints
- Perform production transformations


## 1. Runtime parameters

This manual preparation notebook uses the same infrastructure identifiers as `lab05_00_setup`.

`seed_snapshot_count` controls how many initial status files are written before the first pipeline run.

The snapshots are intentionally separate files because the streaming side of this lab is a **growing file source**, not one JSON file repeatedly overwritten.


In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "Catalog")
dbutils.widgets.text("schema", "parvinbadalov", "Schema")
dbutils.widgets.text("volume_name", "lab05_lakeflow", "Managed volume")
dbutils.widgets.text(
    "streaming_volume_name",
    "lab05_lakeflow_streaming",
    "Streaming external volume"
)
dbutils.widgets.text("seed_snapshot_count", "3", "Seed snapshot count")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume_name = dbutils.widgets.get("volume_name").strip()
streaming_volume_name = dbutils.widgets.get(
    "streaming_volume_name"
).strip()
seed_snapshot_count = int(
    dbutils.widgets.get("seed_snapshot_count").strip()
)

if seed_snapshot_count < 1:
    raise ValueError("seed_snapshot_count must be at least 1.")

volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
streaming_volume_path = (
    f"/Volumes/{catalog}/{schema}/{streaming_volume_name}"
)
status_landing_path = (
    f"{streaming_volume_path}/landing/station_status"
)
reference_path = f"{volume_path}/reference"

station_information_path = (
    f"{reference_path}/station_information.json"
)

print(f"Managed volume       : {volume_path}")
print(f"Streaming volume     : {streaming_volume_path}")
print(f"Status landing path  : {status_landing_path}")
print(f"Reference path       : {reference_path}")
print(f"Seed snapshots       : {seed_snapshot_count}")


Managed volume       : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow
Streaming volume     : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming
Status landing path  : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status
Reference path       : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/reference
Seed snapshots       : 3


## 2. Verify that setup was completed

`lab05_00_setup` must be run before this notebook.

Source preparation should **not** create infrastructure as a side effect. If the expected volume directories are missing, this notebook fails early and directs the user back to setup.


In [0]:
required_paths = [
    status_landing_path,
    reference_path,
]

for path in required_paths:
    try:
        dbutils.fs.ls(path)
        print(f"✅ Found: {path}")
    except Exception as exc:
        raise RuntimeError(
            f"Required path does not exist: {path}. "
            "Run lab05_00_setup first."
        ) from exc


✅ Found: /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status
✅ Found: /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/reference


## 3. Discover the current Citi Bike GBFS feed URLs

Instead of hardcoding the underlying `station_status` and `station_information` endpoints, this notebook starts from Citi Bike's GBFS discovery feed and resolves the current URLs by feed name.

This is more robust because the discovery feed is the source of truth for the system's current GBFS endpoints.


In [0]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import requests


GBFS_DISCOVERY_URL = (
    "https://gbfs.citibikenyc.com/gbfs/2.3/gbfs.json"
)


def get_json(url: str) -> dict:
    """Download JSON and fail clearly for HTTP/network errors."""
    response = requests.get(
        url,
        timeout=30,
        headers={
            "User-Agent": "Databricks-Lab05-Lakeflow/1.0"
        },
    )
    response.raise_for_status()
    return response.json()


discovery = get_json(GBFS_DISCOVERY_URL)

feeds = {
    feed["name"]: feed["url"]
    for feed in discovery["data"]["en"]["feeds"]
}

required_feeds = {
    "station_information",
    "station_status",
}

missing_feeds = sorted(required_feeds - set(feeds))

if missing_feeds:
    raise ValueError(
        "Required GBFS feeds were not found: "
        + ", ".join(missing_feeds)
    )

station_information_url = feeds["station_information"]
station_status_url = feeds["station_status"]

print("✅ GBFS discovery successful")
print()
print("Resolved feeds:")
print(f"station_information: {station_information_url}")
print(f"station_status     : {station_status_url}")


✅ GBFS discovery successful

Resolved feeds:
station_information: https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_information.json
station_status     : https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_status.json


## 4. Download the batch/reference source

`station_information.json` is treated as the Lab 05 batch/reference source.

The raw GBFS payload is preserved rather than flattening it during source preparation. Flattening and dataset creation belong to the declarative pipeline.

Rerunning this notebook refreshes the reference file with the latest station metadata.


In [0]:
station_information_payload = get_json(
    station_information_url
)

reference_file = Path(station_information_path)

reference_file.write_text(
    json.dumps(
        station_information_payload,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

reference_size = reference_file.stat().st_size

print(
    "✅ Saved station_information reference:"
)
print(station_information_path)
print(f"Size: {reference_size:,} bytes")


✅ Saved station_information reference:
/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/reference/station_information.json
Size: 743,268 bytes


## 5. Create initial streaming snapshots

For the streaming side, every API call writes a **new timestamped file**.

Important behavior:

```text
poll #1 → station_status_<timestamp>.json
poll #2 → station_status_<timestamp>.json
poll #3 → station_status_<timestamp>.json
```

No existing snapshot is overwritten.

These are **seed snapshots**, not historical backfill data. The GBFS feed may return the same state for closely spaced requests because the upstream feed has its own refresh interval. That is acceptable here: the goal is to establish the growing-file ingestion pattern.

Later, `tools/citibike_status_producer.py` will implement the same single-shot snapshot behavior independently of this notebook.


In [0]:
created_snapshot_paths = []

for snapshot_number in range(
    1,
    seed_snapshot_count + 1,
):
    payload = get_json(station_status_url)

    timestamp = datetime.now(
        timezone.utc
    ).strftime("%Y%m%dT%H%M%S%fZ")

    snapshot_path = (
        f"{status_landing_path}/"
        f"station_status_{timestamp}.json"
    )

    Path(snapshot_path).write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    created_snapshot_paths.append(snapshot_path)

    print(
        f"✅ Snapshot {snapshot_number}: "
        f"{snapshot_path}"
    )

    # Small delay prevents filename collisions.
    # Citi Bike's upstream state may still be unchanged.
    if snapshot_number < seed_snapshot_count:
        time.sleep(1)


✅ Snapshot 1: /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213131918281Z.json
✅ Snapshot 2: /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213133823532Z.json
✅ Snapshot 3: /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213135085111Z.json


## 6. Verify the raw files

At this point the volume should contain:

```text
lab05_lakeflow/
├── landing/
│   └── station_status/
│       ├── station_status_<timestamp>.json
│       ├── station_status_<timestamp>.json
│       └── ...
└── reference/
    └── station_information.json
```


In [0]:
status_files = [
    file_info
    for file_info in dbutils.fs.ls(
        status_landing_path
    )
    if file_info.name.endswith(".json")
]

reference_files = [
    file_info
    for file_info in dbutils.fs.ls(
        reference_path
    )
    if file_info.name == "station_information.json"
]

print(
    f"Status JSON files found : {len(status_files)}"
)
print(
    f"Reference file found    : {bool(reference_files)}"
)

display(
    spark.createDataFrame(
        [
            (
                f.name,
                f.path,
                f.size,
            )
            for f in status_files
        ],
        ["file_name", "path", "size_bytes"],
    )
)


Status JSON files found : 9
Reference file found    : True


file_name,path,size_bytes
station_status_20260816T212444297598Z.json,dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T212444297598Z.json,1073830
station_status_20260816T212446460935Z.json,dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T212446460935Z.json,1073830
station_status_20260816T212447763131Z.json,dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T212447763131Z.json,1073830
station_status_20260816T212835258322Z.json,dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T212835258322Z.json,1073819
station_status_20260816T212836904157Z.json,dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T212836904157Z.json,1073819
station_status_20260816T212838170639Z.json,dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T212838170639Z.json,1073819
station_status_20260816T213131918281Z.json,dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213131918281Z.json,1074008
station_status_20260816T213133823532Z.json,dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213133823532Z.json,1074008
station_status_20260816T213135085111Z.json,dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213135085111Z.json,1074008


## 7. Parse and flatten `station_information` for profiling

The raw GBFS document has a top-level envelope and a nested array:

```text
data.stations[]
```

For profiling only, explode that array into one row per station.

This flattened DataFrame is **not written as a table**. The actual Bronze dataset will later be declared in `pipeline/bronze.py`.


In [0]:
from pyspark.sql import functions as F


station_information_raw_df = (
    spark.read
    .option("multiLine", True)
    .json(station_information_path)
)

station_information_df = (
    station_information_raw_df
    .select(
        F.explode("data.stations").alias("station")
    )
    .select("station.*")
)

print(
    f"Station information rows: "
    f"{station_information_df.count():,}"
)

station_information_df.printSchema()

display(
    station_information_df.limit(10)
)


Station information rows: 2,509
root
 |-- capacity: long (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- name: string (nullable = true)
 |-- region_id: string (nullable = true)
 |-- rental_uris: struct (nullable = true)
 |    |-- android: string (nullable = true)
 |    |-- ios: string (nullable = true)
 |-- short_name: string (nullable = true)
 |-- station_id: string (nullable = true)



capacity,lat,lon,name,region_id,rental_uris,short_name,station_id
25,40.740821207823245,-73.93147185444832,34 St & 48 Ave,71,"List(https://bkn.lft.to/lastmile_qr_scan, https://bkn.lft.to/lastmile_qr_scan)",6110.01,f523c481-ba41-44ba-96e1-0c431fbea558
18,40.82188,-73.907521,Cauldwell Ave & E 161 St,71,"List(https://bkn.lft.to/lastmile_qr_scan, https://bkn.lft.to/lastmile_qr_scan)",7952.02,493850bd-125b-49f5-932b-bf999504fb06
50,40.74848023448579,-73.98255586624146,E 35 St & Madison Ave,71,"List(https://bkn.lft.to/lastmile_qr_scan, https://bkn.lft.to/lastmile_qr_scan)",6398.08,2637fed8-1e2a-460b-83eb-3eb32ace0f4e
31,40.70188858913366,-74.01089876890182,South St & Broad St,71,"List(https://bkn.lft.to/lastmile_qr_scan, https://bkn.lft.to/lastmile_qr_scan)",4920.13,c00ef46d-fcde-48e2-afbd-0fb595fe3fa7
55,40.75513557,-73.98658032,Broadway & W 41 St,71,"List(https://bkn.lft.to/lastmile_qr_scan, https://bkn.lft.to/lastmile_qr_scan)",6560.01,66dc292c-0aca-11e7-82f6-3863bb44ef7c
21,40.67709,-73.9008,Fulton St & Williams Ave,71,"List(https://bkn.lft.to/lastmile_qr_scan, https://bkn.lft.to/lastmile_qr_scan)",4120.02,038b7369-9c6e-4eb2-8569-f7461443302f
28,40.6772744,-73.98282002,Union St & 4 Ave,71,"List(https://bkn.lft.to/lastmile_qr_scan, https://bkn.lft.to/lastmile_qr_scan)",4175.15,66de0d91-0aca-11e7-82f6-3863bb44ef7c
22,40.67428,-73.9503,Prospect Pl & Nostrand Ave,71,"List(https://bkn.lft.to/lastmile_qr_scan, https://bkn.lft.to/lastmile_qr_scan)",4025.02,475c44e1-31e3-4c60-b071-af2ca3f7618c
25,40.706732,-73.961241,Lee Ave & Taylor St,71,"List(https://bkn.lft.to/lastmile_qr_scan, https://bkn.lft.to/lastmile_qr_scan)",5053.01,1871698223344654662
56,40.731135726910246,-74.00815155572697,Morton St & Greenwich St,71,"List(https://bkn.lft.to/lastmile_qr_scan, https://bkn.lft.to/lastmile_qr_scan)",5772.05,549b5bc4-b949-4eeb-8ffb-2215e1e49892


## 8. Parse and flatten all current `station_status` snapshots

All JSON snapshots currently present in the landing directory are read together.

A source-file column is added so we can distinguish which snapshot produced each row.


In [0]:
station_status_raw_df = (
    spark.read
    .option("multiLine", True)
    .json(f"{status_landing_path}/*.json")
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
    )
)

station_status_df = (
    station_status_raw_df
    .select(
        F.explode("data.stations").alias("station"),
        "_source_file",
    )
    .select(
        "station.*",
        "_source_file",
    )
)

print(
    f"Station status rows across snapshots: "
    f"{station_status_df.count():,}"
)

station_status_df.printSchema()

display(
    station_status_df.limit(10)
)


Station status rows across snapshots: 22,581
root
 |-- is_installed: long (nullable = true)
 |-- is_renting: long (nullable = true)
 |-- is_returning: long (nullable = true)
 |-- last_reported: long (nullable = true)
 |-- num_bikes_available: long (nullable = true)
 |-- num_bikes_disabled: long (nullable = true)
 |-- num_docks_available: long (nullable = true)
 |-- num_docks_disabled: long (nullable = true)
 |-- num_ebikes_available: long (nullable = true)
 |-- num_scooters_available: long (nullable = true)
 |-- num_scooters_unavailable: long (nullable = true)
 |-- station_id: string (nullable = true)
 |-- vehicle_types_available: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- count: long (nullable = true)
 |    |    |-- vehicle_type_id: string (nullable = true)
 |-- _source_file: string (nullable = false)



is_installed,is_renting,is_returning,last_reported,num_bikes_available,num_bikes_disabled,num_docks_available,num_docks_disabled,num_ebikes_available,num_scooters_available,num_scooters_unavailable,station_id,vehicle_types_available,_source_file
1,1,1,1786915679,20,3,2,0,13,0,0,f523c481-ba41-44ba-96e1-0c431fbea558,"List(List(7, 1), List(13, 2))",dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213131918281Z.json
1,1,1,1786915680,0,4,13,0,0,0,0,493850bd-125b-49f5-932b-bf999504fb06,"List(List(0, 1), List(0, 2))",dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213131918281Z.json
1,1,1,1786915679,18,3,27,0,6,0,0,2637fed8-1e2a-460b-83eb-3eb32ace0f4e,"List(List(12, 1), List(6, 2))",dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213131918281Z.json
1,1,1,1786915679,28,0,3,0,17,0,0,c00ef46d-fcde-48e2-afbd-0fb595fe3fa7,"List(List(11, 1), List(17, 2))",dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213131918281Z.json
1,1,1,1786915679,4,3,48,0,0,0,0,66dc292c-0aca-11e7-82f6-3863bb44ef7c,"List(List(4, 1), List(0, 2))",dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213131918281Z.json
1,1,1,1786915678,7,3,11,0,3,0,0,038b7369-9c6e-4eb2-8569-f7461443302f,"List(List(4, 1), List(3, 2))",dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213131918281Z.json
1,1,1,1786915679,19,7,0,1,0,0,0,66de0d91-0aca-11e7-82f6-3863bb44ef7c,"List(List(19, 1), List(0, 2))",dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213131918281Z.json
1,1,1,1786915679,0,5,16,0,0,0,0,475c44e1-31e3-4c60-b071-af2ca3f7618c,"List(List(0, 1), List(0, 2))",dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213131918281Z.json
1,1,1,1786915698,19,2,4,0,14,0,0,1871698223344654662,"List(List(5, 1), List(14, 2))",dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213131918281Z.json
1,1,1,1786915698,49,5,0,0,6,0,0,549b5bc4-b949-4eeb-8ffb-2215e1e49892,"List(List(43, 1), List(6, 2))",dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status/station_status_20260816T213131918281Z.json


## 9. Profile `station_information`

We check the fields most important for later reference-data quality rules and joins:

- row count
- distinct `station_id`
- null `station_id`
- duplicate `station_id`
- null station name
- null/negative capacity

The results will guide the explicit schema and expectation definitions created in later phases.


In [0]:
info_total_rows = station_information_df.count()

info_distinct_station_ids = (
    station_information_df
    .select("station_id")
    .distinct()
    .count()
)

info_null_station_ids = (
    station_information_df
    .filter(F.col("station_id").isNull())
    .count()
)

info_duplicate_station_ids = (
    station_information_df
    .groupBy("station_id")
    .count()
    .filter(
        F.col("station_id").isNotNull()
        & (F.col("count") > 1)
    )
    .count()
)

info_null_names = (
    station_information_df
    .filter(F.col("name").isNull())
    .count()
    if "name" in station_information_df.columns
    else None
)

info_invalid_capacity = (
    station_information_df
    .filter(
        F.col("capacity").isNull()
        | (F.col("capacity") < 0)
    )
    .count()
    if "capacity" in station_information_df.columns
    else None
)

info_profile = [
    ("total_rows", info_total_rows),
    (
        "distinct_station_ids",
        info_distinct_station_ids,
    ),
    (
        "null_station_ids",
        info_null_station_ids,
    ),
    (
        "duplicate_station_ids",
        info_duplicate_station_ids,
    ),
    ("null_station_names", info_null_names),
    (
        "null_or_negative_capacity",
        info_invalid_capacity,
    ),
]

display(
    spark.createDataFrame(
        info_profile,
        ["metric", "value"],
    )
)


metric,value
total_rows,2509
distinct_station_ids,2509
null_station_ids,0
duplicate_station_ids,0
null_station_names,0
null_or_negative_capacity,0


## 10. Profile `station_status`

For the streaming source, check:

- total rows across all snapshot files
- number of snapshot files
- distinct `station_id`
- null `station_id`
- negative bike availability
- negative dock availability
- `last_reported` range

A station can legitimately appear in multiple files because each file represents a different observation time. Therefore, duplicate `station_id` values **across snapshots are expected** and are not treated as bad data.


In [0]:
status_total_rows = station_status_df.count()

status_snapshot_count = (
    station_status_df
    .select("_source_file")
    .distinct()
    .count()
)

status_distinct_station_ids = (
    station_status_df
    .select("station_id")
    .distinct()
    .count()
)

status_null_station_ids = (
    station_status_df
    .filter(F.col("station_id").isNull())
    .count()
)

status_negative_bikes = (
    station_status_df
    .filter(F.col("num_bikes_available") < 0)
    .count()
    if "num_bikes_available"
    in station_status_df.columns
    else None
)

status_negative_docks = (
    station_status_df
    .filter(F.col("num_docks_available") < 0)
    .count()
    if "num_docks_available"
    in station_status_df.columns
    else None
)

status_profile = [
    ("total_rows", status_total_rows),
    ("snapshot_files", status_snapshot_count),
    (
        "distinct_station_ids",
        status_distinct_station_ids,
    ),
    (
        "null_station_ids",
        status_null_station_ids,
    ),
    (
        "negative_bikes_available",
        status_negative_bikes,
    ),
    (
        "negative_docks_available",
        status_negative_docks,
    ),
]

display(
    spark.createDataFrame(
        status_profile,
        ["metric", "value"],
    )
)

if "last_reported" in station_status_df.columns:
    display(
        station_status_df.select(
            F.min("last_reported").alias(
                "min_last_reported"
            ),
            F.max("last_reported").alias(
                "max_last_reported"
            ),
        )
    )


metric,value
total_rows,22581
snapshot_files,9
distinct_station_ids,2509
null_station_ids,0
negative_bikes_available,0
negative_docks_available,0


min_last_reported,max_last_reported
86400,1786915818


## 11. Validate the natural join key

The project depends on `station_id` being a reliable relationship between the live status feed and station metadata.

This check compares the **distinct current status station IDs** against `station_information`.

Ideally, every status station has a matching information record. If unmatched stations exist, they are displayed for investigation rather than silently discarded.


In [0]:
status_station_ids_df = (
    station_status_df
    .select("station_id")
    .filter(F.col("station_id").isNotNull())
    .distinct()
)

information_station_ids_df = (
    station_information_df
    .select("station_id")
    .filter(F.col("station_id").isNotNull())
    .distinct()
)

unmatched_status_station_ids_df = (
    status_station_ids_df
    .join(
        information_station_ids_df,
        on="station_id",
        how="left_anti",
    )
)

unmatched_station_count = (
    unmatched_status_station_ids_df.count()
)

matched_station_count = (
    status_station_ids_df.count()
    - unmatched_station_count
)

status_station_count = status_station_ids_df.count()

join_coverage_pct = (
    matched_station_count
    / status_station_count
    * 100
    if status_station_count
    else 0.0
)

join_profile = [
    (
        "distinct_status_station_ids",
        str(status_station_count),
    ),
    (
        "matched_station_ids",
        str(matched_station_count),
    ),
    (
        "unmatched_station_ids",
        str(unmatched_station_count),
    ),
    (
        "join_coverage_pct",
        f"{join_coverage_pct:.2f}%",
    ),
]

display(
    spark.createDataFrame(
        join_profile,
        ["metric", "value"],
    )
)

if unmatched_station_count > 0:
    print(
        "⚠️ Some status stations have no matching "
        "station_information record."
    )
    display(
        unmatched_status_station_ids_df.limit(20)
    )
else:
    print(
        "✅ Every status station_id has a matching "
        "station_information record."
    )


metric,value
distinct_status_station_ids,2509
matched_station_ids,2509
unmatched_station_ids,0
join_coverage_pct,100.00%


✅ Every status station_id has a matching station_information record.


## 12. Final source-preparation validation

The notebook finishes with explicit checks.

These checks confirm only **source readiness**. They do not replace Silver expectations or final pipeline validation.

Required conditions:
- `station_information.json` exists
- at least one status snapshot exists
- both sources contain rows
- neither source has null `station_id` values
- status data has at least one station that matches the reference dataset


In [0]:
source_checks = [
    (
        "reference_file_exists",
        len(reference_files) == 1,
    ),
    (
        "status_snapshot_exists",
        status_snapshot_count >= 1,
    ),
    (
        "station_information_has_rows",
        info_total_rows > 0,
    ),
    (
        "station_status_has_rows",
        status_total_rows > 0,
    ),
    (
        "information_station_id_not_null",
        info_null_station_ids == 0,
    ),
    (
        "status_station_id_not_null",
        status_null_station_ids == 0,
    ),
    (
        "join_has_matches",
        matched_station_count > 0,
    ),
]

source_validation_df = spark.createDataFrame(
    [
        (
            check_name,
            "PASS" if passed else "FAIL",
        )
        for check_name, passed in source_checks
    ],
    ["check", "result"],
)

display(source_validation_df)

failed_source_checks = [
    name
    for name, passed in source_checks
    if not passed
]

if failed_source_checks:
    raise AssertionError(
        "Lab 05 source preparation failed: "
        + ", ".join(failed_source_checks)
    )

print("✅ LAB 05 SOURCE PREPARATION PASSED")
print()
print(
    f"Reference stations     : "
    f"{info_total_rows:,}"
)
print(
    f"Status snapshot files  : "
    f"{status_snapshot_count:,}"
)
print(
    f"Status rows             : "
    f"{status_total_rows:,}"
)
print(
    f"Join coverage           : "
    f"{join_coverage_pct:.2f}%"
)


check,result
reference_file_exists,PASS
status_snapshot_exists,PASS
station_information_has_rows,PASS
station_status_has_rows,PASS
information_station_id_not_null,PASS
status_station_id_not_null,PASS
join_has_matches,PASS


✅ LAB 05 SOURCE PREPARATION PASSED

Reference stations     : 2,509
Status snapshot files  : 9
Status rows             : 22,581
Join coverage           : 100.00%


## What this proved

After a successful run:

### Batch source
A raw `station_information.json` reference file exists and contains station metadata.

### Streaming source
The landing folder contains multiple immutable `station_status_<timestamp>.json` snapshot files that can be incrementally consumed by Auto Loader.

### Joinability
Both datasets expose `station_id`, and the notebook measures the actual match coverage before we build the Silver enrichment.

### Separation of responsibilities
This notebook prepares and profiles **raw source files only**. It does not create pipeline tables.

---

## Recommended evidence

Capture one screenshot showing:
- the source profile tables, and
- the successful join-key coverage.

Suggested README image:

```text
01_source_profile.png
```

---

## Next step

After this notebook passes, continue with:

```text
tools/citibike_status_producer.py
```

The producer will extract the single-shot status-snapshot logic from this notebook into a reusable script:

```text
poll current GBFS station_status
        ↓
write one timestamped JSON file
        ↓
exit
```

After that we will create the explicit source schemas and start `pipeline/bronze.py`.
